In [ ]:
!pip install transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Modèle léger pour Colab Free
model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

print("✅ Model loaded successfully!")

In [ ]:
!pip install python-docx

In [ ]:
from docx import Document

def read_docx(file_path):
    doc = Document(file_path)
    text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
    return text

In [ ]:
!pip install python-docx transformers accelerate bitsandbytes

In [ ]:
from google.colab import files

print("Upload all files (PDS, PSAC, SCS, SDP, SDS, SRS, SVP, SYS-ARCH, prompt)")
uploaded = files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/M1_Inputs')

In [ ]:
# 1. Va dans ton dossier
import os
os.chdir('/content/M1_Inputs')

# 2. Vérifie les fichiers
!ls -la

In [ ]:
!pip install python-docx transformers accelerate bitsandbytes -q

In [ ]:
from docx import Document

def read_docx(filename):
    doc = Document(filename)
    return "\n".join([p.text for p in doc.paragraphs])

In [ ]:
# Lire le system prompt M1
with open("m1_system_prompt.txt", "r") as f:
    system_prompt = f.read()

# Lire les documents d'entrée
pds = read_docx("PDS_STM32F303_I-SCEET.docx")
srs = read_docx("SRS-001_I-SCEET.docx")
sds = read_docx("SDS-001_I-SCEET.docx")
sys_arch = read_docx("SYS-ARCH-001_I-SCEET.docx")

print("✅ Tous les documents lus")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Chargement du modèle en 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True
)
print("✅ Modèle chargé")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configuration 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Chargement du modèle en 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Modèle chargé")

In [ ]:
print("Chargement du modèle (sans 4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Modèle chargé")

In [ ]:
user_prompt = f"""
Here are the input documents for M1:

# PDS (Product Design Specification)
{pds[:3000]}

# SRS-001 (Software Requirements Standards)
{srs[:2000]}

# SDS-001 (Software Design Specification)
{sds[:2000]}

# SYS-ARCH-001 (System Architecture)
{sys_arch[:2000]}

Based on these documents, generate SRD-001 with:
- SECTION 1: System Architecture (copy verbatim from SYS-ARCH-001)
- SECTION 2: High Level Requirements (following the HLR format in the system prompt)
"""

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False)
print("✅ Prompt construit")
print(f"Longueur : {len(text)} caractères")

In [ ]:
print("Génération de SRD-001 en cours...")

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=4096,
    temperature=0.1,
    do_sample=False
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("✅ Génération terminée")